# Block 7 — LAB: Evaluating Models — Threshold Adjustment & Class Imbalance
### Advanced Machine Learning — M&T Bank

Loads `bank_marketing_features.csv` (unchanged since Block 3). Same feature prep and train/test split as Blocks
4-6. No new CSV comes out of this lab either.

**Where Block 6 left off:** at the default 0.5 threshold, our logistic regression catches only 20.5% of clients
who'd actually subscribe (recall), while being right 68.1% of the time when it does flag someone (precision).

**Part 1 — Threshold adjustment:** the 0.5 cutoff isn't a law of nature — it's a default. Sweep it and see what
opens up.

**Part 2 — Class imbalance via `class_weight`:** a second lever that changes *training* instead of the decision
rule after the fact. Compare it head-to-head with threshold tuning.

**Part 3 — A fair-lending-style check:** any lever that changes who gets flagged is worth checking for who it
changes it *for*. Age is an ECOA-protected basis — we check flag rates across our own `age` column as a live
example of the technique.

Look for `# TODO` — that's where your code goes. Each task has a hint; ask if you get stuck.


## Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score,
)
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.max_columns", 30)

NAVY = "#251E4E"
PINK = "#FF1675"
GRAY = "#6b7280"
ORANGE = "#FF7B01"

# Paste the raw GitHub URL for bank_marketing_features.csv below, then remove the leading '#':
# df = pd.read_csv("PASTE_RAW_GITHUB_URL_HERE", sep=";")
print(df.shape)
df.head()


In [ ]:
# Same prep and split as Blocks 4-6, for numeric continuity
feature_cols = [c for c in df.columns if c not in ("education", "y", "duration")]
bool_cols = [c for c in df[feature_cols].columns if df[c].dtype == bool]

numeric_to_scale = [
    "age", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
    "education_rank", "month_sin", "month_cos", "campaign_intensity",
    "emp.var.rate_denoised", "cons.price.idx_denoised", "cons.conf.idx_denoised",
    "euribor3m_denoised", "nr.employed_denoised",
]

X = df[feature_cols].copy()
for c in bool_cols:
    X[c] = X[c].astype(int)
y = df["y"].values

# TODO: train_test_split X, y -> X_train, X_test, y_train, y_test (test_size=0.2, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = None, None, None, None

scaler = StandardScaler()
X_train_s = X_train.copy()
X_test_s = X_test.copy()
# TODO: fit_transform the training numeric columns, transform the test numeric columns


baseline = LogisticRegression(max_iter=2000, solver="lbfgs", C=0.5)
baseline.fit(X_train_s, y_train)
proba = baseline.predict_proba(X_test_s)[:, 1]

print("Baseline @ default 0.5 threshold:")
pred_default = (proba >= 0.5).astype(int)
print(f"  precision={precision_score(y_test, pred_default):.3f}  "
      f"recall={recall_score(y_test, pred_default):.3f}  "
      f"f1={f1_score(y_test, pred_default):.3f}")


# Part 1 — Threshold Adjustment

### Task 1.1 — Sweep the threshold

`predict()` defaults to flagging a client whenever `predict_proba >= 0.5`. That 0.5 is just a default — nothing
about the model requires it. Sweep a range of thresholds and recompute precision, recall, and F1 at each.

**Hint:** `np.arange(0.05, 0.96, 0.05)`, and for each threshold `t`, `(proba >= t).astype(int)`.


In [ ]:
thresholds = np.arange(0.05, 0.96, 0.05)
sweep_rows = []
for t in thresholds:
    # TODO: compute pred_t = (proba >= t).astype(int), then precision/recall/f1
    pred_t = None
    sweep_rows.append({
        "threshold": round(t, 2),
        "precision": None,  # TODO
        "recall": None,     # TODO
        "f1": None,          # TODO
        "n_flagged": None,   # TODO: pred_t.sum()
    })
sweep = pd.DataFrame(sweep_rows)
sweep.round(3)


### Task 1.2 — Plot precision and recall against threshold

**Hint:** two lines on one axis, `sweep["threshold"]` on the x-axis.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(sweep["threshold"], sweep["precision"], color=NAVY, linewidth=2, marker="o", markersize=4, label="Precision")
ax.plot(sweep["threshold"], sweep["recall"], color=PINK, linewidth=2, marker="o", markersize=4, label="Recall")
ax.plot(sweep["threshold"], sweep["f1"], color=ORANGE, linewidth=2, linestyle="--", label="F1")
ax.axvline(0.5, color=GRAY, linestyle=":", linewidth=1.5, label="Default threshold")

# TODO: find best_idx = the row index where sweep["f1"] is maximized (hint: .idxmax())
best_idx = None
best_t = sweep.loc[best_idx, "threshold"]
ax.axvline(best_t, color=GRAY, linestyle="-", linewidth=1, alpha=0.4)
ax.scatter([best_t], [sweep.loc[best_idx, "f1"]], color=ORANGE, s=80, zorder=5)

ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision, Recall, and F1 as the Threshold Moves", color=NAVY, fontweight="bold")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"F1-maximizing threshold: {best_t:.2f}  (F1={sweep.loc[best_idx, 'f1']:.3f}, "
      f"vs. {f1_score(y_test, pred_default):.3f} at the default 0.5)")


### Task 1.3 — Compare confusion matrices: default vs. tuned threshold

**Hint:** `confusion_matrix(y_test, (proba >= best_t).astype(int))`.


In [ ]:
# TODO: pred_tuned = (proba >= best_t).astype(int)
pred_tuned = None
cm_default = confusion_matrix(y_test, pred_default)
cm_tuned = confusion_matrix(y_test, pred_tuned)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.3))
for ax, cm, title in zip(axes, [cm_default, cm_tuned], [f"Threshold = 0.50 (default)", f"Threshold = {best_t:.2f} (F1-optimal)"]):
    im = ax.imshow(cm, cmap="RdPu")
    labels = [["TN", "FP"], ["FN", "TP"]]
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{labels[i][j]}\n{cm[i, j]:,}", ha="center", va="center",
                    fontsize=12, color="white" if cm[i, j] > cm.max() / 2 else NAVY, fontweight="bold")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred: No", "Pred: Yes"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Actual: No", "Actual: Yes"])
    ax.set_title(title, color=NAVY, fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()


**Question to answer before moving on:** how many more clients does the tuned threshold catch (TP), and how many
more wasted calls does it cost (FP)? Is that trade worth it?


# Part 2 — Class Imbalance via `class_weight`

### Task 2.1 — Refit with `class_weight="balanced"`

A different lever: instead of moving the decision boundary after training, change what the *training* process
penalizes. `class_weight="balanced"` makes mistakes on the minority class ("yes") cost proportionally more during
fitting, so the model itself shifts toward catching more of them.

**Hint:** same `LogisticRegression(max_iter=2000, solver="lbfgs", C=0.5)`, add `class_weight="balanced"`.


In [ ]:
# TODO: create and fit balanced_model with class_weight="balanced"
balanced_model = None

# TODO: get predict_proba[:, 1] -> proba_balanced, and predict() -> pred_balanced
proba_balanced = None
pred_balanced = None

print("Balanced model @ default 0.5 threshold:")
print(f"  precision={precision_score(y_test, pred_balanced):.3f}  "
      f"recall={recall_score(y_test, pred_balanced):.3f}  "
      f"f1={f1_score(y_test, pred_balanced):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, proba_balanced):.4f}  "
      f"(baseline was {roc_auc_score(y_test, proba):.4f})")


### Task 2.2 — Three-way comparison

Put all three side by side: the untouched baseline, the class-weighted model (still at 0.5), and the baseline with
its threshold tuned instead.


In [ ]:
comparison = pd.DataFrame({
    "Approach": ["Baseline @ 0.50", "class_weight=balanced @ 0.50", f"Baseline @ tuned {best_t:.2f}"],
    "Precision": [
        precision_score(y_test, pred_default),
        precision_score(y_test, pred_balanced),
        precision_score(y_test, pred_tuned),
    ],
    "Recall": [
        recall_score(y_test, pred_default),
        recall_score(y_test, pred_balanced),
        recall_score(y_test, pred_tuned),
    ],
    "F1": [
        f1_score(y_test, pred_default),
        f1_score(y_test, pred_balanced),
        f1_score(y_test, pred_tuned),
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, proba),
        roc_auc_score(y_test, proba_balanced),
        roc_auc_score(y_test, proba),
    ],
}).round(4)
comparison


**Question to answer before moving on:** which of the three rows has the best F1? Does that match your intuition
about which technique "should" work better — retraining with class weights, or simply moving the threshold?


# Part 3 — A Fair-Lending-Style Check

**Important framing:** this dataset is a term-deposit *marketing* model, not a credit or lending decision, so ECOA
doesn't directly apply to it. But the analytical technique — checking whether a model's selection rate differs
sharply across a protected class, using a screen like the "four-fifths rule" — is exactly what would be applied to
an actual credit, pricing, or underwriting model, and age is one of ECOA's enumerated protected bases. This is a
live rehearsal of that check, not a real regulatory finding about this dataset.

### Task 3.1 — Flag rate by age band, at two thresholds

**Hint:** `pd.cut(X_test["age"], bins=..., labels=..., include_lowest=True)`, same bin edges as earlier blocks:
`[17, 25, 35, 45, 55, 65, 98]`.


In [ ]:
# TODO: create age_band_test using pd.cut on X_test["age"]
age_bins = [17, 25, 35, 45, 55, 65, 98]
age_labels = ["18-25", "26-35", "36-45", "46-55", "56-65", "66+"]
age_band_test = None

flag_rates = pd.DataFrame({
    "Default (0.50)": pd.Series(pred_default, index=X_test.index).groupby(age_band_test, observed=True).mean() * 100,
    f"Tuned ({best_t:.2f})": pd.Series(pred_tuned, index=X_test.index).groupby(age_band_test, observed=True).mean() * 100,
}).round(1)
flag_rates


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(flag_rates))
width = 0.38
ax.bar(x - width/2, flag_rates["Default (0.50)"], width, label="Default (0.50)", color=NAVY)
ax.bar(x + width/2, flag_rates[f"Tuned ({best_t:.2f})"], width, label=f"Tuned ({best_t:.2f})", color=PINK)
ax.set_xticks(x)
ax.set_xticklabels(flag_rates.index)
ax.set_ylabel("% of clients flagged")
ax.set_title("Flag Rate by Age Band — Both Thresholds", color=NAVY, fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


### Task 3.2 — The four-fifths rule

A common screening heuristic: if the lowest-flagged group's rate is under 80% of the highest-flagged group's rate,
that's a signal worth investigating further (not, by itself, proof of unlawful discrimination — legitimate business
justification and less-discriminatory-alternative analysis come next in a real review).

**Hint:** `flag_rates.min() / flag_rates.max()` for each column.


In [ ]:
# TODO: compute the min/max ratio for each column of flag_rates
fourfifths = None
print("Min/max flag-rate ratio by threshold (0.80 = the common four-fifths screening line):")
print(fourfifths)


**Question to answer before moving on:** did tuning the threshold in Part 1 meaningfully close this gap, or just
shift the overall flag rate up? What does that tell you about whether threshold adjustment is a fairness fix?


## Recap

- The 0.5 classification threshold is a convention, not a requirement — sweeping it and picking a point on the
  precision/recall trade-off is a free, post-hoc lever that doesn't require retraining.
- `class_weight="balanced"` changes the training objective instead; compare both rather than assuming either wins
  by default.
- ROC-AUC stays essentially flat across threshold/weighting changes, because none of these techniques change how
  well the model *ranks* clients — only where you choose to draw the line.
- Any lever that changes who gets flagged is worth checking for who it changes it *for* — and threshold adjustment
  alone is not a fix for a representational disparity that's baked into the model's features.
- No new CSV from this block — `bank_marketing_features.csv` carries forward unchanged.

**Up next — Block 8:** Evaluating Models — Advanced Metrics + Day 1 Tie-Together.
